# Flight Delay Intelligence - Exploratory Data Analysis

Source: PostgreSQL table `flight_features` (built from US DOT BTS On-Time Performance data).

Sections:
1. Delay rate by airline
2. Delay rate by airport
3. Delay by hour of day
4. Delay causes
5. Route-level analysis

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from database import get_engine

sns.set_theme(style='whitegrid')
engine = get_engine()

## 1. Delay rate by airline

In [ ]:
query = '''
SELECT
    op_unique_carrier AS airline,
    COUNT(*) AS total_flights,
    100.0 * SUM(CASE WHEN is_arrival_delayed THEN 1 ELSE 0 END) / COUNT(*) AS delay_rate_pct
FROM flight_features
GROUP BY op_unique_carrier
ORDER BY delay_rate_pct DESC;
'''
airline_delays = pd.read_sql(query, engine)
airline_delays.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=airline_delays, x='airline', y='delay_rate_pct', ax=ax, color='steelblue')
ax.set_title('Arrival Delay Rate by Airline')
ax.set_ylabel('Delay rate (%)')
plt.xticks(rotation=45)
plt.tight_layout();

## 2. Delay rate by airport

In [ ]:
query = '''
SELECT
    origin AS airport,
    COUNT(*) AS total_flights,
    100.0 * SUM(CASE WHEN is_arrival_delayed THEN 1 ELSE 0 END) / COUNT(*) AS delay_rate_pct
FROM flight_features
GROUP BY origin
HAVING COUNT(*) >= 1000
ORDER BY delay_rate_pct DESC
LIMIT 20;
'''
airport_delays = pd.read_sql(query, engine)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=airport_delays, y='airport', x='delay_rate_pct', ax=ax, color='indianred')
ax.set_title('Top 20 Airports by Delay Rate (>= 1k flights)')
ax.set_xlabel('Delay rate (%)')
plt.tight_layout();

## 3. Delay by hour of day

In [ ]:
query = '''
SELECT
    dep_hour,
    AVG(arr_delay) AS avg_arr_delay,
    100.0 * SUM(CASE WHEN is_arrival_delayed THEN 1 ELSE 0 END) / COUNT(*) AS delay_rate_pct
FROM flight_features
GROUP BY dep_hour
ORDER BY dep_hour;
'''
hourly = pd.read_sql(query, engine)

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=hourly, x='dep_hour', y='delay_rate_pct', marker='o', ax=ax)
ax.set_title('Arrival Delay Rate by Departure Hour')
ax.set_xlabel('Departure hour (24h)')
ax.set_ylabel('Delay rate (%)')
plt.tight_layout();

## 4. Delay causes

In [ ]:
query = '''
SELECT
    SUM(carrier_delay)        AS carrier,
    SUM(weather_delay)        AS weather,
    SUM(nas_delay)            AS nas,
    SUM(security_delay)       AS security,
    SUM(late_aircraft_delay)  AS late_aircraft
FROM flight_features;
'''
causes = pd.read_sql(query, engine).T.reset_index()
causes.columns = ['cause', 'total_minutes']
causes = causes.sort_values('total_minutes', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=causes, x='cause', y='total_minutes', ax=ax, color='teal')
ax.set_title('Total Delay Minutes by Cause')
ax.set_ylabel('Total minutes')
plt.tight_layout();

## 5. Route-level analysis

In [ ]:
query = '''
SELECT
    route,
    COUNT(*) AS total_flights,
    100.0 * SUM(CASE WHEN is_arrival_delayed THEN 1 ELSE 0 END) / COUNT(*) AS delay_rate_pct,
    AVG(arr_delay) AS avg_arr_delay
FROM flight_features
GROUP BY route
HAVING COUNT(*) >= 100
ORDER BY delay_rate_pct DESC
LIMIT 20;
'''
worst_routes = pd.read_sql(query, engine)
worst_routes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=worst_routes, y='route', x='delay_rate_pct', ax=ax, color='darkorange')
ax.set_title('Top 20 Worst Routes by Delay Rate')
ax.set_xlabel('Delay rate (%)')
plt.tight_layout();

### Takeaways

- Delay rates vary meaningfully by airline and airport.
- Late-day flights (afternoon / evening) tend to be delayed more often than morning flights.
- `late_aircraft_delay` and `carrier_delay` dominate total delay minutes.
- A small set of routes accounts for a disproportionate share of the worst delays.

These findings drive the feature choices in `02_modeling.ipynb`.